# 06 · What drives the forecast

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USER/chronos2-in-practice/blob/main/notebooks/06_explainability.ipynb)

Companion notebook for *Chronos-2 in Practice (3/5)*.

**What you will learn**

- Four model-agnostic ways to ask what a forecast is using, in increasing order of cost:
  ablation, permutation, exact Shapley values and occlusion.
- Which of the covariates from notebook `05_covariates` the model actually relies on, and
  why ablation and Shapley can disagree.
- Which part of the past matters, using the context segments of Hertel et al. (2026).
- A what-if: move a known-future covariate and watch the forecast move with it.

Runtime: ~10 min on a GPU (FAST_MODE: ~2 min).

In [ ]:
# On Google Colab, clone the companion repo and install it (Colab's own PyTorch is kept).
# Locally, `uv sync` already installed everything and this cell does nothing.
import sys

if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/YOUR_GITHUB_USER/chronos2-in-practice.git
    %cd chronos2-in-practice
    %pip install -q .
    import importlib

    importlib.invalidate_caches()

In [ ]:
import time

import pandas as pd
from huggingface_hub.utils import logging as hub_logging
from IPython.display import Image, display
from transformers.utils import logging as transformers_logging

from c2p.config import get_device, get_paths, is_fast_mode, set_seed
from c2p.data import (
    CALENDAR_COLUMNS,
    add_calendar_features,
    spain_energy_frame,
    validate_and_prepare,
)
from c2p.eval import iter_windows
from c2p.experiments import FREQ, PRICE, TSO_COVARIATES, day_ahead_config
from c2p.explain import (
    IMPORTANCE_COL,
    PLAYER_COL,
    SHAPLEY_COL,
    ablation_importance,
    context_segments,
    covariate_shapley,
    occlusion,
    permutation_importance,
    stack_windows,
    temporal_shapley,
)
from c2p.models import Chronos2Forecaster, load_chronos2
from c2p.plotting import (
    plot_benchmarks,
    plot_metric_vs_param,
    plot_scenarios,
    save_figure,
    set_style,
)
from c2p.results import load_results, run_metadata, save_results
from c2p.schema import ID_COL, TIMESTAMP_COL

FAST_MODE = is_fast_mode(default=False)
PATHS = get_paths(fast_mode=FAST_MODE)
set_seed()
DEVICE = get_device()
set_style()
hub_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

for key, value in run_metadata(device=DEVICE, fast_mode=FAST_MODE).items():
    print(f"{key:>20}: {value}")

## 1. Chronos-2 ships no explainer, and that is fine

There is no `.explain()`, no feature-importance attribute, no built-in attribution. What there
is instead is the thing every attribution method actually needs: a cheap way to re-run the model
with part of its input removed. Zero-shot inference costs one forward pass, so switching an input
off and measuring the damage is affordable in a way it never is for a model you had to train.

Every technique below is **model-agnostic**: none of them looks inside the network. That is a
deliberate choice, not a limitation to apologise for. Attention weights are tempting — the group
attention layer is right there — but a decade of work on attention as explanation (Jain &
Wallace, 2019) says attention weights are not a faithful account of what a model used, and
Chronos-2's group attention has had no such validation at all. Inputs you can switch off give
answers you can check.

The four methods, in increasing order of cost and fidelity:

| Method | Cost | What it tells you | What it misses |
| --- | --- | --- | --- |
| **Ablation** | `n + 1` forecasts | What drops when one covariate is removed | Interactions: two redundant covariates both look useless |
| **Permutation** | `n x repeats` | What drops when one covariate is scrambled | Same, plus it feeds the model impossible inputs |
| **Shapley** | `2**n` forecasts | Each player's average marginal contribution over all coalitions | Nothing in principle; affordable only for a handful of players |
| **Occlusion** | one per position | Which *part of the past* matters | Attribution to inputs, not to mechanism |

Shapley values are exact here, not sampled: the trick that makes them affordable is grouping the
inputs into a few meaningful players rather than approximating the average.

In [ ]:
COLUMNS = [ID_COL, TIMESTAMP_COL, PRICE, *TSO_COVARIATES]
price, report = validate_and_prepare(spain_energy_frame()[COLUMNS], freq=FREQ, target=PRICE)
price = add_calendar_features(price)
CALENDAR = tuple(CALENDAR_COLUMNS)
KNOWN = (*CALENDAR, *TSO_COVARIATES)

# The players: calendar as one group, each operator forecast on its own.
PLAYERS = {
    "calendar": CALENDAR,
    "load forecast": ("load_forecast_tso",),
    "solar forecast": ("solar_forecast_tso",),
    "wind forecast": ("wind_onshore_forecast_tso",),
}
print("Players:", {name: len(columns) for name, columns in PLAYERS.items()})

Explaining one window would explain one day. Instead a handful of day-ahead windows from the
2018 test year are *stacked*: each window's series becomes its own series, so one `predict` call
covers them all and every number below is an average over genuinely different days.

In [ ]:
N_WINDOWS = 3 if FAST_MODE else 12
CONFIG = day_ahead_config(FAST_MODE, target=PRICE, known_covariates=KNOWN, max_context=2048)
CONFIG = type(CONFIG)(
    prediction_length=CONFIG.prediction_length,
    n_windows=N_WINDOWS,
    freq=CONFIG.freq,
    season_length=CONFIG.season_length,
    step=24 * 28,  # four weeks apart, so the windows land in different seasons
    quantile_levels=CONFIG.quantile_levels,
    target=CONFIG.target,
    known_covariates=CONFIG.known_covariates,
    max_context=CONFIG.max_context,
)
windows = list(iter_windows(price, CONFIG))
context, future, actuals = stack_windows(windows)
print(
    f"{len(windows)} windows stacked into {context[ID_COL].nunique()} series, "
    f"{len(context):,} context rows, {len(future):,} future rows"
)
print("Cutoffs:", [str(window.cutoff.date()) for window in windows])

pipeline = load_chronos2(device=DEVICE)
chronos = Chronos2Forecaster(pipeline)

## 2. The cheap questions: ablation and permutation

Both score the same way here: the increase in WQL when a covariate is switched off, so a positive
number means the covariate was helping.

In [ ]:
start = time.perf_counter()
shared = dict(
    covariates=PLAYERS,
    prediction_length=CONFIG.prediction_length,
    freq=FREQ,
    target=PRICE,
    actuals=actuals,
)
ablation = ablation_importance(chronos, context, future, **shared).assign(method="Ablation")
permutation = permutation_importance(
    chronos, context, future, n_repeats=2 if FAST_MODE else 5, **shared
).assign(method="Permutation")
cheap = pd.concat([ablation, permutation], ignore_index=True)
print(f"Ablation + permutation in {time.perf_counter() - start:.0f} s")
print("Saved", save_results(cheap, "06_covariate_importance", PATHS).name)

In [ ]:
cheap = load_results("06_covariate_importance", PATHS)
summary = (
    cheap.groupby([PLAYER_COL, "method"], as_index=False)[IMPORTANCE_COL]
    .mean()
    .rename(columns={PLAYER_COL: "model", "method": "benchmark"})
)
top = summary[summary["benchmark"] == "Ablation"].sort_values(IMPORTANCE_COL, ascending=False)
title = (
    f"Removing the {top.iloc[0]['model']} costs the most; "
    f"the {top.iloc[-1]['model']} costs the least"
)
fig = plot_benchmarks(
    summary,
    value=IMPORTANCE_COL,
    highlight=str(top.iloc[0]["model"]),
    xlabel="WQL increase when the covariate is switched off (higher = more useful)",
    title=title,
)
(figure_path,) = save_figure(fig, "06_covariate_importance", PATHS.figures)
display(Image(filename=figure_path, width=800))
print(
    summary.pivot_table(index="model", columns="benchmark", values=IMPORTANCE_COL)
    .round(5)
    .to_string()
)

## 3. The honest question: exact Shapley values

Ablation asks "what happens without this covariate, when all the others are present". That is one
coalition out of sixteen, and it is the one most likely to understate a covariate that another can
substitute for. Shapley values average each player's marginal contribution over *every* subset of
the others, which is the only way redundancy and complementarity show up.

With four players that is 16 forecasts — and because coalitions share work, far fewer than 16
separate calls.

In [ ]:
start = time.perf_counter()
explanation = covariate_shapley(
    chronos,
    context,
    future,
    covariates=PLAYERS,
    prediction_length=CONFIG.prediction_length,
    freq=FREQ,
    target=PRICE,
    value="wql_reduction",
    actuals=actuals,
)
print(
    f"{len(explanation.players)} players -> {explanation.n_forecasts} coalition forecasts "
    f"in {explanation.n_calls} calls, {explanation.seconds:.0f} s"
)
print("Saved", save_results(explanation.shapley, "06_covariate_shapley", PATHS).name)

In [ ]:
shapley = load_results("06_covariate_shapley", PATHS)
per_player = (
    shapley.groupby(PLAYER_COL, as_index=False)[SHAPLEY_COL]
    .mean()
    .rename(columns={PLAYER_COL: "model"})
    .assign(benchmark="Shapley value")
)
ranked = per_player.sort_values(SHAPLEY_COL, ascending=False)
best, worst = ranked.iloc[0], ranked.iloc[-1]
title = (
    f"{best['model']} contributes most ({best[SHAPLEY_COL]:+.4f} WQL), "
    f"{worst['model']} least ({worst[SHAPLEY_COL]:+.4f})"
)
fig = plot_benchmarks(
    per_player,
    value=SHAPLEY_COL,
    highlight=str(best["model"]),
    xlabel="Shapley value: share of the WQL reduction attributable to the covariate",
    title=title,
)
(figure_path,) = save_figure(fig, "06_covariate_shapley", PATHS.figures)
display(Image(filename=figure_path, width=800))
print(
    per_player[["model", SHAPLEY_COL]]
    .sort_values(SHAPLEY_COL, ascending=False)
    .to_string(index=False)
)

## 4. Which part of the past?

The same machinery applies to the target's own history. Following Hertel et al. (2026), the
context is cut into four segments — the last day, the rest of the week, the rest of four weeks and
everything older — and each is a player that a coalition can switch off by setting those target
values to NaN.

This answers a question the context sweep of notebook `03` could only answer in aggregate: not
"how much history helps" but *which* history.

In [ ]:
SEGMENTS = context_segments((24, 168, 672))
print("Segments:", [segment.name for segment in SEGMENTS])

temporal = temporal_shapley(
    chronos,
    context,
    future,
    segments=(24, 168, 672),
    prediction_length=CONFIG.prediction_length,
    freq=FREQ,
    target=PRICE,
    value="wql_reduction",
    actuals=actuals,
)
print(
    f"{len(temporal.players)} players -> {temporal.n_forecasts} coalition forecasts "
    f"in {temporal.n_calls} calls, {temporal.seconds:.0f} s"
)
print("Saved", save_results(temporal.shapley, "06_context_shapley", PATHS).name)

In [ ]:
context_shapley = load_results("06_context_shapley", PATHS)
per_segment = (
    context_shapley.groupby(PLAYER_COL, as_index=False)[SHAPLEY_COL]
    .mean()
    .rename(columns={PLAYER_COL: "model"})
    .assign(benchmark="Shapley value")
)
ranked = per_segment.sort_values(SHAPLEY_COL, ascending=False)
best, worst = ranked.iloc[0], ranked.iloc[-1]
title = (
    f"The most recent day matters most ({best['model']}: {best[SHAPLEY_COL]:+.4f} WQL)"
    if best["model"] == "lags 1-24"
    else f"{best['model']} matters most ({best[SHAPLEY_COL]:+.4f} WQL), {worst['model']} least"
)
fig = plot_benchmarks(
    per_segment,
    value=SHAPLEY_COL,
    highlight=str(best["model"]),
    xlabel="Shapley value: share of the WQL reduction attributable to the segment",
    title=title,
)
(figure_path,) = save_figure(fig, "06_context_shapley", PATHS.figures)
display(Image(filename=figure_path, width=800))
print(
    per_segment[["model", SHAPLEY_COL]]
    .sort_values(SHAPLEY_COL, ascending=False)
    .to_string(index=False)
)

Occlusion asks the same thing at finer resolution and far lower cost: slide a one-day NaN window
back through the context and record the damage at each position. It cannot attribute (every
position is scored with all the others present), but it shows the shape.

In [ ]:
occluded = occlusion(
    chronos,
    context,
    future,
    window=24,
    stride=24,
    max_lag=168 if FAST_MODE else 672,
    prediction_length=CONFIG.prediction_length,
    freq=FREQ,
    target=PRICE,
    actuals=actuals,
)
print("Saved", save_results(occluded, "06_occlusion", PATHS).name)

occluded = load_results("06_occlusion", PATHS)
by_lag = (
    occluded.groupby("first_lag", as_index=False)[IMPORTANCE_COL]
    .mean()
    .assign(part="One day hidden")
)
worst = by_lag.sort_values(IMPORTANCE_COL, ascending=False).iloc[0]
ax = plot_metric_vs_param(
    by_lag,
    x="first_lag",
    y=IMPORTANCE_COL,
    hue="part",
    colors={},
    logx=True,
    title=f"Hiding the day at lag {int(worst['first_lag'])} h hurts most",
    xlabel="First hidden lag (hours before the cutoff)",
    ylabel="WQL increase",
)
(figure_path,) = save_figure(ax.figure, "06_occlusion", PATHS.figures)
display(Image(filename=figure_path, width=800))

## 5. What-if: move a covariate and watch

Attribution says what mattered on average. A what-if says what *this* forecast would have been
under a different input — which is usually the question a planner is really asking. Because the
operator's wind forecast is a known-future covariate, changing it is a one-line intervention.

This is a sensitivity analysis, not a causal claim: it shows how the model responds, which is
useful precisely because it is checkable against domain knowledge. More wind in the Spanish
market should push the day-ahead price *down*.

In [ ]:
ONE = sorted(context[ID_COL].unique())[-1]  # the most recent stacked window
WIND = "wind_onshore_forecast_tso"
one_context = context[context[ID_COL] == ONE]
one_future = future[future[ID_COL] == ONE]
one_actuals = actuals[actuals[ID_COL] == ONE]

scenarios = {}
rows = []
for label, factor in (("Operator's forecast", 1.0), ("Wind x1.5", 1.5), ("Wind x0.5", 0.5)):
    shifted = one_future.assign(**{WIND: one_future[WIND] * factor})
    forecast = chronos.predict(
        one_context,
        prediction_length=CONFIG.prediction_length,
        quantile_levels=list(CONFIG.quantile_levels),
        freq=FREQ,
        target=PRICE,
        future_covariates=shifted,
    )
    scenarios[label] = (one_context, forecast)
    rows.append(forecast.assign(scenario=label, wind_factor=factor))
what_if = pd.concat(rows, ignore_index=True)
print("Saved", save_results(what_if, "06_what_if", PATHS).name)

means = what_if.groupby("scenario")["predictions"].mean()
base = means["Operator's forecast"]
for label in ("Wind x1.5", "Wind x0.5"):
    print(
        f"  {label}: mean day-ahead price {means[label]:.2f} EUR/MWh "
        f"({means[label] - base:+.2f} vs the operator's forecast)"
    )

In [ ]:
what_if = load_results("06_what_if", PATHS)
fig = plot_scenarios(
    {label: (one_context, what_if[what_if["scenario"] == label]) for label in scenarios},
    actuals=one_actuals,
    target=PRICE,
    history=168,
    freq=FREQ,
    sharey=True,
    title="Halving the wind forecast lifts the predicted price, and vice versa",
    ylabel="Day-ahead price (EUR/MWh)",
)
(figure_path,) = save_figure(fig, "06_what_if", PATHS.figures)
display(Image(filename=figure_path, width=800))

## 6. Summary, and what none of this proves

- **Ablation is the cheapest useful answer** and usually enough to rank covariates. Where it
  disagrees with the Shapley values, the disagreement is the finding: it means two inputs can
  stand in for each other.
- **Shapley values over a handful of grouped players are affordable** for a zero-shot model, and
  they are exact rather than sampled. Grouping is what makes them possible — four players, sixteen
  coalitions.
- **The same machinery explains the past**, segment by segment, and occlusion gives the shape at
  finer resolution for the price of one forecast per position.
- **What-ifs answer the planner's question** and are checkable against domain knowledge, which
  attribution numbers are not.

What none of it establishes is mechanism. These are statements about the model's *inputs*, not
about electricity markets: if two covariates are correlated, attribution will split their credit
in a way that reflects the model, not the physics. And the group attention weights are not in this
notebook on purpose — reading them as explanation is exactly the step the literature warns
against.

**Next:** article 4 and [`07_finetune_lora`](07_finetune_lora.ipynb) — when adapting the weights
beats feeding better inputs.